# VideoDB Search V2 Guide

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/search/search_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Build a searchable video, then move from precise retrieval to intelligent search and playable evidence.

In this guide you will:

- create transcript and structured visual indexes
- use `semantic_search()`, `query()`, and `aggregate()` directly
- let Search, DeepSearch, and Ask plan retrieval from natural language
- target an entire index or one semantic field
- inspect, play, compile, and embed timestamped results

## 1. Install dependencies

This guide uses the published VideoDB SDK.

In [1]:
!pip install -q videodb python-dotenv

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 161.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 158.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 195.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.1/187.1 kB 234.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 226.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 240.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 204.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 167.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you hav

## 2. Connect to VideoDB

Set `VIDEO_DB_API_KEY` in Colab secrets or your environment. If it is missing, the cell asks for it securely.

In [3]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"], base_url="https://api.dev.videodb.io")
collection = conn.get_collection()

print("Connected to VideoDB")
print("Collection:", collection.id)

Connected to VideoDB
Collection: c-0b1cc2b2-b945-43ba-8e99-0b6d0ac1480e


## 3. Choose a video

The default is a short sample clip. To use an existing video, comment out the upload and provide its ID instead.

In [4]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

video = collection.upload(url=VIDEO_URL)

# To use an existing video instead:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Video:", video.id)
video.play()

Video: m-z-019f85c9-cd3f-7950-aa07-79ae8c1a0676


## 4. Create searchable video understanding

Search works over indexes. First, create two reusable analyzer outputs:

- `transcript` for spoken words
- `scene` for structured visual understanding

The VLM uses frames as its source of visual evidence and receives the transcript as supporting context. Its nested schema gives us stable fields for semantic search, filtering, and aggregation.

In [5]:
scene_schema = {
    "scene_description": "string",
    "activity": {
        "type": "enum",
        "values": ["conversation", "using_device", "walking", "object_interaction", "other"],
    },
    "setting": {
        "type": "object",
        "fields": {
            "location_type": {
                "type": "enum",
                "values": ["office", "home", "outdoor", "vehicle", "other"],
            },
            "environment": {
                "type": "enum",
                "values": ["indoor", "outdoor", "unknown"],
            },
        },
    },
    "object_descriptions": {
        "type": "array",
        "min_items": 0,
        "max_items": 4,
        "items": {
            "name": "string",
            "description": "string",
        },
    },
}

understanding = video.understand(
    analyzers=[
        {
            "type": "spoken_words",
            "name": "transcript",
            "config": {"language": "en"},
        },
        {
            "type": "vlm",
            "name": "scene",
            "inputs": ["transcript"],
            "sampling": {"strategy": "uniform", "frame_count": 6},
            "config": {
                "model": "pro",
                "prompt": (
                    "Analyze the sampled frames using the frames as the primary source of visual evidence. "
                    "Write scene_description as one clear, specific sentence describing what happens. "
                    "Classify the main activity and setting. For each notable visible object, return a short "
                    "name and a concise visual description of its appearance or role in the scene. "
                    "Use the transcript only to disambiguate the activity. Do not infer visual details from "
                    "the transcript, and do not include objects that are not visible.\n\n"
                ),
                "schema": scene_schema,
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding:", understanding.id)
print("Status:", understanding.status)

Understanding: und_ffe78b72228f4534
Status: queued


## 5. Wait for analyzers

Understanding runs asynchronously. Wait before indexing its outputs.

In [6]:
understanding.wait_until_complete(timeout=3600, poll_interval=15)

for analyzer in understanding.list_analyzers():
    print(analyzer.name, analyzer.type, analyzer.status)

transcript_analyzer = understanding.get_analyzer("transcript")
scene_analyzer = understanding.get_analyzer("scene")

transcript speech_transcription done
scene vlm done


## 6. Create retrieval-ready indexes

The transcript index uses derived defaults. The scene index explicitly declares which fields support semantic search, filtering, aggregation, and sorting.

Unique names make the notebook safe to rerun in the same collection.

In [7]:
from datetime import datetime, timezone

run_suffix = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
transcript_index_name = f"transcript_{run_suffix}"
scene_index_name = f"scene_{run_suffix}"

transcript_index = video.index(
    name=transcript_index_name,
    source=transcript_analyzer,
)

scene_index = video.index(
    name=scene_index_name,
    source=scene_analyzer,
    use_for=["semantic", "query", "aggregate"],
    fields={
        "semantic": [
            "scene_description",
            "activity",
            "setting.location_type",
            "object_descriptions.description",
        ],
        # "text": ["scene_description"],
        "filter": [
            "activity",
            "setting.location_type",
            "setting.environment",
            "object_descriptions.name",
        ],
        "aggregate": [
            "activity",
            "setting.location_type",
            "setting.environment",
            "object_descriptions.name",
        ]
    },
)

print("Transcript index:", transcript_index.index_id, transcript_index.name)
print("Scene index:", scene_index.index_id, scene_index.name)

Transcript index: d8b2b78ca43e44a5 transcript_20260721175400
Scene index: 43952379b61b4b50 scene_20260721175400


## 7. Wait for semantic indexes

Structured query and aggregation can become available before embeddings finish. This guide waits for `ready` so every example, including semantic search, can run.

In [8]:
import time

READY_STATUSES = {"ready", "done", "completed"}
FAILED_STATUSES = {"failed", "error"}


def wait_for_index(index, timeout=900, poll_interval=10):
    deadline = time.time() + timeout

    while True:
        current = video.get_index(index_id=index.index_id)
        status = str(current.status or "").lower()
        print(current.name, status)

        if status in READY_STATUSES:
            return current
        if status in FAILED_STATUSES:
            raise RuntimeError(f"Index {current.name} failed: {current.error}")
        if time.time() > deadline:
            raise TimeoutError(f"Timed out waiting for {current.name}")

        time.sleep(poll_interval)


transcript_index = wait_for_index(transcript_index)
scene_index = wait_for_index(scene_index)

transcript_20260721175400 building
transcript_20260721175400 building
transcript_20260721175400 building
transcript_20260721175400 ready
scene_20260721175400 building
scene_20260721175400 ready


## 8. Inspect the index contract

`fields` shows how each field can be used. `field_schema` adds the type and supported operators for each field.

In [9]:
print("Scene capabilities:", scene_index.use_for)
print("Scene fields:", scene_index.fields)
print()

for field, schema in scene_index.field_schema.items():
    print(
        field,
        "type=", schema.type,
        "groups=", schema.groups,
        "operators=", schema.operators,
    )

Scene capabilities: ['semantic', 'query', 'aggregate']
Scene fields: {'aggregate': ['activity', 'setting.location_type', 'setting.environment', 'object_descriptions.name'], 'filter': ['activity', 'setting.location_type', 'setting.environment', 'object_descriptions.name'], 'semantic': ['scene_description', 'activity', 'setting.location_type', 'object_descriptions.description']}

activity type= string groups= ['semantic', 'filter', 'aggregate'] operators= ['==', '!=', 'contains', 'in', 'exists']
object_descriptions.description type= string_array groups= ['semantic'] operators= []
object_descriptions.name type= string_array groups= ['filter', 'aggregate'] operators= ['==', '!=', 'contains', 'in', 'exists']
scene_description type= text groups= ['semantic'] operators= []
setting.environment type= string groups= ['filter', 'aggregate'] operators= ['==', '!=', 'contains', 'in', 'exists']
setting.location_type type= string groups= ['semantic', 'filter', 'aggregate'] operators= ['==', '!=', 'co

## 9. Choose a search method

| Goal | Method | Index selection |
|---|---|---|
| Let VideoDB interpret the request | `search()` | VideoDB chooses |
| Search meaning directly | `semantic_search()` | Zero, one, or many semantic indexes |
| Apply exact conditions | `query()` | Exactly one index |
| Count or group values | `aggregate()` | Exactly one index |
| Investigate over multiple steps | DeepSearch | VideoDB chooses |
| Generate an answer | `ask()` | VideoDB chooses |

## 10. Search from natural language

Use `search()` when you know the goal and want VideoDB to choose the retrieval strategy.

In [10]:
search_response = video.search(
    query="Find the moment when an electronic device is thrown away",
    top_k=5,
    return_fields=[scene_index_name, transcript_index_name],
)

print("Response type:", search_response.response_type)
for shot in search_response:
    print(shot.start, shot.end, shot.search_score, shot.text)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:18<00:00,  5.36it/s]

Response type: shots
38.789 39.623 0.5186052299999999 A person in a plaid shirt stands at a desk and drops a white cable or earphones into a small trash bin in front of the desk while multiple monitors and desk items are visible.


### Play one moment

`play()` opens the matching moment. `generate_stream()` returns its playable HLS URL.

In [36]:
search_response[0].play()

### Compile several moments

Direct semantic search and query return a `SearchResult`, which can compile its shots into one stream.

In [38]:
search_response.results.compile()

'https://dseetlpshk2tb.cloudfront.net/v3/published/manifests/d1e30789-2a00-4e13-bd2d-50a48e6b05b0.m3u8'

## 11. Search a semantic index

Passing an index name searches all semantic fields inside that index.

In [11]:
semantic_results = video.semantic_search(
    query="someone handling an electronic device",
    index_names=[scene_index_name],
    top_k=5,
    score_threshold=0.2,
    return_fields=[scene_index_name],
)

for shot in semantic_results:
    print(shot.start, shot.end, shot.search_score, shot.text)

37.037 38.789 0.6010772 bearded man with glasses and long hair wearing a plaid shirt, standing and holding a device multiple large monitors arranged across the desk displaying code and terminal windows with teal backlighting small green handheld device with white cables attached, held in the man's hands white cardboard box on the desk with a blue label and small logo on the side},{
33.951 35.202 0.54499787 A person in a plaid shirt leans over a desk full of monitors and reaches toward a small black device and other items on the desk.
14.473 19.52 0.5213468 Adult male with beard and glasses wearing a blue plaid shirt, standing and looking down at a device. Handheld phone held in both hands near waist level, being looked at by the man. Multiple widescreen monitors arranged side-by-side and stacked, displaying code and terminal output with teal accents. Black metal stand or rack supporting several vertically oriented monitors on the right side.
8.133 11.136 0.5000774299999999 A person sit

### Target one semantic field

Append a field path to search only that field. This request searches `setting.location_type` instead of all semantic fields in the scene index.

In [12]:
setting_results = video.semantic_search(
    query="inside a home",
    index_names=[f"{scene_index_name}.setting.location_type"],
    top_k=5,
)

for shot in setting_results:
    print(shot.start, shot.end, shot.search_score, shot.text)

88.088 89.381 0.4338022 home
71.113 79.079 0.4338022 home
93.969 97.014 0.4338022 home
90.591 93.969 0.4338022 home
89.381 90.591 0.4338022 home


### Combine meaning with a filter

Semantic search can also require an exact indexed condition.

In [13]:
filtered_semantic_results = video.semantic_search(
    query="people talking",
    index_names=[scene_index_name],
    filter=[
        {"field": "setting.environment", "op": "==", "value": "indoor"},
    ],
    top_k=5,
)

for shot in filtered_semantic_results:
    print(shot.start, shot.end, shot.search_score)

27.361 32.532 0.58414078
35.202 37.037 0.58414078
52.302 53.72 0.58414078
32.532 33.951 0.58414078
53.72 55.472 0.58414078


### Search one nested object field

Target the descriptions inside `object_descriptions` when the search should focus on how visible objects look or function.

In [15]:
object_results = video.semantic_search(
    query="a small electronic device connected by a cable",
    index_names=[f"{scene_index_name}.object_descriptions.description"],
    top_k=5,
    return_fields=[scene_index_name]
)

for shot in object_results:
    print(shot.start, shot.end, shot.search_score, shot.text)

37.037 38.789 0.38068366 bearded man with glasses and long hair wearing a plaid shirt, standing and holding a device multiple large monitors arranged across the desk displaying code and terminal windows with teal backlighting small green handheld device with white cables attached, held in the man's hands white cardboard box on the desk with a blue label and small logo on the side},{
84.501 88.088 0.34440726 Man in a purple-striped sweater and red pants standing in the center facing toward the camera. Bearded man with long hair wearing a red plaid shirt standing to the right, holding a cable. Curly-haired man in a dark green jacket entering from the left, partially visible. Several stacked boxes and gadget packages piled on a low surface near the left foreground.
2.211 5.172 0.3369438 Person seen from behind wearing a gray sweater with a checkered shirt collar, walking down the aisle away from the camera. Multiple wooden desks arranged in clusters with computers, cables, and small desk 

## 12. Query exact conditions

Use `query()` when you know the index and field values. A list of conditions is an implicit AND.

In [16]:
conversation_results = video.query(
    index_name=scene_index_name,
    filter=[
        {"field": "activity", "op": "==", "value": "conversation"},
        {"field": "setting.location_type", "op": "in", "value": ["home", "office"]},
    ],
    limit=20,
    return_fields=[scene_index_name],
)

for shot in conversation_results:
    print(shot.start, shot.end, shot.metadata)

5.172 6.757 {'activity': 'conversation', 'object_descriptions': [{'description': 'Seated adult male in a light button-up shirt looking up and slightly to the right', 'name': 'man'}, {'description': 'Silver laptop on an angled stand at left foreground with its screen facing the man', 'name': 'laptop'}, {'description': 'Silver all-in-one desktop monitor near center of the desk with an Apple logo on the back', 'name': 'iMac monitor'}, {'description': 'Adjustable white desk lamp positioned over the workspace pointing downward', 'name': 'desk lamp'}], 'scene_description': 'A man seated at an office desk looks up toward someone off-frame while working at his computer.', 'setting': {'environment': 'indoor', 'location_type': 'office'}}
6.757 8.133 {'activity': 'conversation', 'object_descriptions': [{'description': 'man wearing a blue sweater and khaki pants walking behind the desk', 'name': 'standing man'}, {'description': 'man in a striped polo seated at the desk looking toward the standing 

### OR and NOT filters

Use explicit boolean groups for alternatives and exclusions.

In [17]:
device_or_object_results = video.query(
    index_name=scene_index_name,
    filter={
        "and": [
            {
                "or": [
                    {"field": "activity", "op": "==", "value": "using_device"},
                    {"field": "activity", "op": "==", "value": "object_interaction"},
                ]
            },
            {
                "not": {"field": "setting.environment", "op": "==", "value": "outdoor"}
            },
        ]
    },
    limit=20,
)

for shot in device_or_object_results:
    print(shot.start, shot.end, shot.text)

8.133 11.136 A person sitting at a multi-monitor workstation in a well-lit room, reaching for and lifting a cardboard box from beside their desk. Multiple monitors show code, and various desk items are visible.
11.136 13.013 A bearded man in a plaid shirt carries a stack of white boxes toward a desk while three coworkers sit at a nearby workstation with multiple monitors and a laptop.
14.473 19.52 A bearded man in a plaid shirt stands in front of several computer monitors and looks down at a smartphone in his hands.
33.951 35.202 A person in a plaid shirt leans over a desk full of monitors and reaches toward a small black device and other items on the desk.
38.789 39.623 A person in a plaid shirt stands at a desk and drops a white cable or earphones into a small trash bin in front of the desk while multiple monitors and desk items are visible.
40.791 45.712 A person in a blue plaid shirt stands at a desk interacting with multiple computer monitors while a small trash can sits on the fl

## 13. Aggregate indexed fields

Aggregation returns rows instead of `Shot` objects. `group_by` is required.

In [18]:
activity_counts = video.aggregate(
    index_name=scene_index_name,
    group_by="activity",
    metric="count",
    limit=20,
)

setting_counts = video.aggregate(
    index_name=scene_index_name,
    group_by="setting.location_type",
    metric="count",
    limit=20,
)

print("Activity counts:", activity_counts)
print("Setting counts:", setting_counts)

Activity counts: [{'activity': 'conversation', 'value': 25.0}, {'activity': 'object_interaction', 'value': 7.0}, {'activity': 'walking', 'value': 2.0}, {'activity': 'using_device', 'value': 2.0}]
Setting counts: [{'setting.location_type': 'office', 'value': 25.0}, {'setting.location_type': 'home', 'value': 11.0}]


## 14. Investigate with DeepSearch

DeepSearch can perform multiple retrieval steps and continue through a session.

In [19]:
deep_response = video.search(
    query="someone handling an electronic device",
    mode="deepsearch",
    top_k=10,
    return_fields="all",
)

print("Session:", deep_response.session_id)
print("Waiting for:", deep_response.waiting_for)
print("Clarification:", deep_response.clarification)

for shot in deep_response.shots:
    print(shot.start, shot.end, shot.text)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:46<00:00,  2.17it/s]

Session: ds-0fe9822b-7c38-4746-89eb-233bc37cfbad
Waiting for: none
Clarification: None
14.473 19.52 scene_20260721175400: using_device
40.791 45.712 scene_20260721175400: using_device
33.951 35.202 scene_20260721175400: object_interaction
98.098 99.808 scene_20260721175400: object_interaction
6.757 8.133 scene_20260721175400: conversation
27.361 32.532 scene_20260721175400: conversation
32.532 33.951 scene_20260721175400: conversation
52.302 53.72 scene_20260721175400: conversation
55.472 59.643 scene_20260721175400: conversation
13.013 14.473 scene_20260721175400: conversation


### Continue or answer a clarification

If DeepSearch asks for more detail, answer by sending another request with the same `session_id`. You can use the same pattern for any follow-up.

In [20]:
if deep_response.clarification:
    print("DeepSearch asks:", deep_response.clarification)

followup_response = video.search(
    query="Focus on the scenes where the device is discarded",
    mode="deepsearch",
    session_id=deep_response.session_id,
    top_k=10,
)

for shot in followup_response.shots:
    print(shot.start, shot.end, shot.text)

  0%|                                                                                                    | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 15. Ask with supporting sources

Ask returns a synthesized answer. `include_sources=True` adds the timestamped moments used as evidence.

In [26]:
answer = video.ask(
    question="What happens to the electronic device",
    top_k=15,
    include_sources=True,
)

print(answer.answer)
print("Sources:", len(answer.sources))

for source in answer.sources:
    print(source.start, source.end, source.text)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:20<00:00,  4.88it/s]

The indexed content shows the device being handled, then a white cable or earphones are dropped into a small trash bin at about 38.8–39.6s. I could not find a direct statement that the electronic device itself is thrown away.
Sources: 3
38.789 39.623 A person in a plaid shirt stands at a desk and drops a white cable or earphones into a small trash bin in front of the desk while multiple monitors and desk items are visible.
37.037 38.789 A bearded man stands in front of a multi-monitor workstation holding a green device with white cables and looks toward the camera as he speaks.
35.202 37.037 Hey, can I have that USB missile launcher?


## 16. Inspect and play results

A `Shot` connects a match to the source video and exact time range.

In [27]:
shots = search_response.shots

if shots:
    shot = shots[0]
    print("Video:", shot.video_id, shot.video_title)
    print("Time:", shot.start, shot.end)
    print("Score:", shot.search_score)
    print("Text:", shot.text)
    print("Evidence:", shot.metadata.get("indexes", {}) if shot.metadata else {})
else:
    print("No shots returned")

Video: m-z-019f85c9-cd3f-7950-aa07-79ae8c1a0676 None
Time: 38.789 39.623
Score: 0.5186052299999999
Text: A person in a plaid shirt stands at a desk and drops a white cable or earphones into a small trash bin in front of the desk while multiple monitors and desk items are visible.
Evidence: {}


### Play one moment

`play()` opens the matching moment. `generate_stream()` returns its playable HLS URL.

In [31]:
shot.play()

In [32]:
stream_url = shot.generate_stream()
print(stream_url)

https://d1zudc7ewmc6ey.cloudfront.net/v1/c094035f-070c-448a-bd83-242f80bbd609.m3u8


### Compile several moments

Direct semantic search and query return a `SearchResult`, which can compile its shots into one stream.

In [33]:
compiled_stream_url = semantic_results.compile()
print(compiled_stream_url)
semantic_results.play()

https://dseetlpshk2tb.cloudfront.net/v3/published/manifests/d9d0c65b-a6f2-493b-ae0c-921387ab2c89.m3u8


For high-level Search or DeepSearch, compile the nested `SearchResult` when the response contains shots.

In [30]:
if search_response.response_type in {"shots", "deepsearch"} and search_response.shots:
    compiled_stream_url = search_response.results.compile()
    print(compiled_stream_url)

https://dseetlpshk2tb.cloudfront.net/v3/published/manifests/d1e30789-2a00-4e13-bd2d-50a48e6b05b0.m3u8


## 17. Search across a collection

Replace `video.*` with `collection.*` when a result may come from any indexed video in the collection. Collection search covers the whole collection.

In [ ]:
collection_results = collection.search(
    query="Find scenes where someone handles an electronic device",
    top_k=10,
)

for shot in collection_results:
    print(shot.video_id, shot.start, shot.end)

## Quick reference

```python
# Intelligent retrieval
video.search(query="...")
video.search(query="...", mode="deepsearch")
video.ask(question="...", include_sources=True)

# Direct retrieval
video.semantic_search(query="...", index_names=["scene"])
video.query(index_name="scene", filter=[...])
video.aggregate(index_name="scene", group_by="activity")

# Result actions
shot.play()
shot.generate_stream()
search_result.compile()
search_result.play()
search_result.get_embed_code()
```

### Index selection

| Method | Selector |
|---|---|
| `semantic_search()` | `index_names` / `index_ids`, zero, one, or many |
| `query()` | `index_name` or `index_id`, exactly one |
| `aggregate()` | `index_name` or `index_id`, exactly one |
| Search, DeepSearch, Ask | No index selectors |